# Additional classical ML methods: no deep learning

Compare RBF SVM, Gaussian Naive Bayes, DBSCAN, and graph-based LabelSpreading against the current validation leader: tied-covariance GMM K=10 plus logistic regression C=1000, sample-wise F1 0.9060. Keep all 150 embeddings; add no year, extra PCA, text, or drug features. All methods are classical machine learning, with no trained neural network.

Use the same 78 validation trials, threshold 0.5, and nonempty fallback. All fitting, graph construction, density-radius selection, scaling, probability calibration, and label generation use training data only. No Kaggle slot is used.

The fixed round compares 18 calibrated RBF-SVM settings and 6 GaussianNB settings on known-only and frozen GMM labels; 6 DBSCAN and 6 LabelSpreading labelers followed by C=1000 logistic regression; and direct graph predictions for the 6 LabelSpreading fits. Finally, cross the best SVM and GaussianNB parameters with the best dataset from each new labeling family. This is a bounded staged search, not proof of a performance ceiling.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
SVM_C = [0.1, 1.0, 10.0]
SVM_GAMMA = [0.001, "scale", 0.03]
NB_SMOOTHING = [1e-9, 1e-3, 1e-1]
DBSCAN_MIN_SAMPLES = [5, 10]
DBSCAN_RADIUS_QUANTILES = [0.25, 0.5, 0.75]
GRAPH_NEIGHBORS = [5, 10, 20]
GRAPH_ALPHA = [0.2, 0.8]
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import DBSCAN
from sklearn.semi_supervised import LabelSpreading
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from scipy.sparse.csgraph import connected_components
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"additional_methods"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


## Reuse fixed classifier, labeling, and evaluation controls

The next helpers reproduce the existing supervised reference. Classifiers use the frozen scaler fitted on 157 known-label trials; clustering/graphs use the existing scaler fitted on 1,171 training trials. Partial targets remain -100 and are excluded independently per reason. Keep the same positive/negative confidence thresholds and capped, class-balanced influence budgets. The observed known class balance is an influence control, not an estimate of population prevalence.

In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
gmm_source=DATA/"experiments/method_comparison/datasets/gmm_tied_k10.csv"
gmm_frame=pd.read_csv(gmm_source,float_precision="round_trip")
assert gmm_frame.ID.tolist()==train.ID.tolist()
gmm_y=gmm_frame[REASONS].to_numpy(dtype=int)
gmm_w=gmm_frame[[f"weight_{r}" for r in REASONS]].to_numpy()
save_dataset("frozen_gmm_tied_k10",gmm_y,gmm_w)
incumbent_path=DATA/"experiments/method_comparison/classifier_models/gmm_tied_k10__logistic_C1000.joblib"
incumbent=joblib.load(incumbent_path)
incumbent_metrics=save_predictions("current_validation_best",predict_reason_models(incumbent,X_valid))
np.testing.assert_allclose(incumbent_metrics["sample_f1"],0.905982905982906,atol=1e-14)
reference_rows.append({"experiment":"current_validation_best","dataset":"frozen_gmm_tied_k10","classifier":"logistic",
                       "parameter":1000.0,"generated_trials":696,**incumbent_metrics,"kaggle_score":np.nan})
protected_hashes.update({str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in [gmm_source,incumbent_path]})
RESULTS=[]
MODELS={"current_validation_best":incumbent}
LABELER_ROWS=[]
GRAPH_MODELS={}
DB_MODELS={}


def fit_additional_classifier(dataset_name,kind,parameters):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert included[known].all() and np.isin(y[included],[0,1]).all()
        key=(kind,json.dumps(parameters,sort_keys=True),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="rbf_svm":
                estimator=SVC(kernel="rbf",C=parameters["C"],gamma=parameters["gamma"],random_state=SEED)
                model=CalibratedClassifierCV(
                    estimator=estimator,method="sigmoid",
                    cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED),
                    ensemble=False,n_jobs=1,
                )
            else:model=GaussianNB(var_smoothing=parameters["var_smoothing"])
            normalized=w[included]*known.sum()/w[included].sum()
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                model.fit(XS[included],y[included],sample_weight=normalized)
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameters":parameters.copy(),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "fitting_weight_normalization":157}


def predict_graph_model(model,raw):
    scaled=model["scaler"].transform(np.asarray(raw))
    fallback=model["fallback_model"].predict_proba(raw)
    probabilities=fallback.copy()
    fallback_count=0
    for j,graph in enumerate(model["graphs"]):
        neighbors=graph.nn_fit.kneighbors(scaled,return_distance=False)
        mass=graph.label_distributions_[neighbors].sum(axis=1)
        total=mass.sum(axis=1)
        reachable=np.isfinite(mass).all(axis=1)&(total>0)
        pos=list(graph.classes_).index(1)
        probabilities[reachable,j]=mass[reachable,pos]/total[reachable]
        fallback_count+=int((~reachable).sum())
    return np.clip(probabilities,0,1),fallback_count


def predict_candidate(model,raw):
    return predict_graph_model(model,raw)[0] if model["kind"]=="graph_direct" else predict_reason_models(model,raw)


def record_candidate(name,model,dataset_name,category,parameters,extra=None):
    probabilities=predict_candidate(model,X_valid)
    metrics=save_predictions(name,probabilities)
    joblib.dump(model,OUT/"classifier_models"/f"{name}.joblib",compress=3)
    row={"experiment":name,"category":category,"classifier":model["kind"],"dataset":dataset_name,
         "parameters":json.dumps(parameters,sort_keys=True),
         "generated_trials":np.nan if model["kind"]=="graph_direct" else DATASETS[dataset_name]["generated_trials"],
         "unlabeled_trials_used":1014 if model["kind"]=="graph_direct" else DATASETS[dataset_name]["generated_trials"],
         **metrics,"delta_vs_current_best":metrics["sample_f1"]-incumbent_metrics["sample_f1"],"kaggle_score":np.nan}
    if extra:row.update(extra)
    RESULTS.append(row);MODELS[name]=model
    print(f"{name}: sample F1={metrics['sample_f1']:.4f}, macro F1={metrics['macro_f1']:.4f}, "
          f"change={row['delta_vs_current_best']:+.4f}",flush=True)
    return row


def targets_from_probabilities(probabilities,support,valid_probability=None):
    valid_probability=np.ones_like(probabilities,dtype=bool) if valid_probability is None else valid_probability
    targets=np.full_like(original_y,-100)
    weights=np.zeros(original_y.shape)
    targets[known]=original_y[known];weights[known]=1.0
    eligible=(~known)[:,None]&(np.asarray(support)>=LABEL_RULE["minimum_support"])[:,None]&valid_probability
    for j in range(5):
        positive=eligible[:,j]&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible[:,j]&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        np_,nn=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not np_ or not nn:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],LABEL_RULE["maximum_generated_weight"]*np_/kp,
                     LABEL_RULE["maximum_generated_weight"]*nn/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/np_;weights[negative,j]=fraction*kn/nn
        assert weights[~known,j].max()<=0.25+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    return targets,weights


def safe_name(value):return str(value).replace(".","p").replace("-","m")

print("Current validation reference:",incumbent_metrics)
print("SVM calibration: sigmoid, 3 stratified training-only folds, final estimator fitted on all included training rows.")

Current validation reference: {'sample_f1': 0.9059829059829059, 'macro_f1': 0.8116263524085356, 'fallback_trials': 4, 'f1_Safety': np.float64(0.9411764705882353), 'f1_Funding': np.float64(0.6666666666666666), 'f1_Efficacy': np.float64(0.8387096774193549), 'f1_Recruitment': np.float64(0.98), 'f1_Other': np.float64(0.631578947368421)}
SVM calibration: sigmoid, 3 stratified training-only folds, final estimator fitted on all included training rows.


## Additional supervised classifiers

RBF SVM has nonlinear boundaries. Use C=0.1,1,10 and gamma=0.001,scale,0.03. Calibrate its probabilities with three stratified training-only folds and sigmoid calibration; validation labels never enter calibration. GaussianNB supports continuous signed features and uses smoothing 1e-9,0.001,0.1. MultinomialNB is not used on these signed embeddings.

Fit each setting on known-only labels and frozen GMM-generated labels, with the same fixed classifier scaler and total fitting weight 157 per reason.

In [4]:
def run_supervised_family(kind):
    if kind=="rbf_svm":
        configs=[{"C":c,"gamma":gamma} for c in SVM_C for gamma in SVM_GAMMA]
    else:configs=[{"var_smoothing":smooth} for smooth in NB_SMOOTHING]
    with threadpool_limits(limits=1):
        for dataset_name in ["known_only","frozen_gmm_tied_k10"]:
            for parameters in configs:
                suffix="_".join(f"{k}{safe_name(v)}" for k,v in parameters.items())
                name=f"{dataset_name}__{kind}_{suffix}"
                model=fit_additional_classifier(dataset_name,kind,parameters)
                record_candidate(name,model,dataset_name,"supervised_comparison",parameters)

In [5]:
run_supervised_family("rbf_svm")

known_only__rbf_svm_C0p1_gamma0p001: sample F1=0.8483, macro F1=0.7034, change=-0.0577


known_only__rbf_svm_C0p1_gammascale: sample F1=0.8077, macro F1=0.6850, change=-0.0983


known_only__rbf_svm_C0p1_gamma0p03: sample F1=0.5684, macro F1=0.4514, change=-0.3376


known_only__rbf_svm_C1p0_gamma0p001: sample F1=0.8483, macro F1=0.6876, change=-0.0577


known_only__rbf_svm_C1p0_gammascale: sample F1=0.8590, macro F1=0.7128, change=-0.0470


known_only__rbf_svm_C1p0_gamma0p03: sample F1=0.7650, macro F1=0.5238, change=-0.1410


... (8 outputs left out)

frozen_gmm_tied_k10__rbf_svm_C1p0_gamma0p03: sample F1=0.6752, macro F1=0.4788, change=-0.2308


frozen_gmm_tied_k10__rbf_svm_C10p0_gamma0p001: sample F1=0.8547, macro F1=0.6125, change=-0.0513


frozen_gmm_tied_k10__rbf_svm_C10p0_gammascale: sample F1=0.8675, macro F1=0.7206, change=-0.0385


frozen_gmm_tied_k10__rbf_svm_C10p0_gamma0p03: sample F1=0.8077, macro F1=0.5408, change=-0.0983


In [6]:
run_supervised_family("gaussian_nb")

known_only__gaussian_nb_var_smoothing1em09: sample F1=0.6389, macro F1=0.5237, change=-0.2671


known_only__gaussian_nb_var_smoothing0p001: sample F1=0.6389, macro F1=0.5237, change=-0.2671


known_only__gaussian_nb_var_smoothing0p1: sample F1=0.7201, macro F1=0.6713, change=-0.1859


frozen_gmm_tied_k10__gaussian_nb_var_smoothing1em09: sample F1=0.6902, macro F1=0.5420, change=-0.2158


frozen_gmm_tied_k10__gaussian_nb_var_smoothing0p001: sample F1=0.6902, macro F1=0.5420, change=-0.2158


frozen_gmm_tied_k10__gaussian_nb_var_smoothing0p1: sample F1=0.7329, macro F1=0.6760, change=-0.1731


## Density-based labels with DBSCAN

Use min_samples=5 or 10. Derive eps from the 25th, 50th, or 75th percentile of training distances to the corresponding nearest neighbor (including self), with a positive floor 1e-6. No validation information chooses these radii. Noise points stay unlabeled for generated targets; original known labels are always preserved.

Estimate cluster reason prevalences with the same prior smoothing and apply the same partial-label and influence budgets. Train C=1000 logistic regression afterward.

In [7]:
def run_dbscan(min_samples):
    with threadpool_limits(limits=1):
        distances=NearestNeighbors(n_neighbors=min_samples,algorithm="brute").fit(XC).kneighbors(XC)[0][:,-1]
        for quantile in DBSCAN_RADIUS_QUANTILES:
            eps=max(float(np.quantile(distances,quantile)),1e-6)
            name=f"dbscan_m{min_samples}_q{round(quantile*100):02d}"
            density=DBSCAN(eps=eps,min_samples=min_samples,metric="euclidean",algorithm="brute",n_jobs=1).fit(XC)
            ids=density.labels_
            unique=np.unique(ids[ids>=0])
            if len(unique):
                responsibilities=np.column_stack([ids==cluster for cluster in unique]).astype(float)
                counts=responsibilities[known].sum(axis=0)
                positive=responsibilities[known].T@original_y[known]
                prior=original_y[known].mean(axis=0)
                rates=(positive+LABEL_RULE["prior_strength"]*prior)/(counts[:,None]+LABEL_RULE["prior_strength"])
                probabilities=responsibilities@rates
                support=responsibilities@counts
                clusters=pd.DataFrame({"cluster":unique,"trials":responsibilities.sum(axis=0),"known_support":counts})
                for j,r in enumerate(REASONS):clusters[f"probability_{r}"]=rates[:,j]
            else:
                probabilities=np.tile(original_y[known].mean(axis=0),(len(train),1));support=np.zeros(len(train))
                clusters=pd.DataFrame(columns=["cluster","trials","known_support"])
            targets,weights=targets_from_probabilities(probabilities,support)
            valid_mask=np.repeat((ids>=0)[:,None],5,axis=1)
            save_dataset(name,targets,weights,np.where(valid_mask,probabilities,np.nan),ids,support)
            assert (weights[(ids<0)&~known]==0).all()
            clusters.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
            joblib.dump({"model":density,"scaler":cluster_scaler,"eps_quantile":quantile,"label_rule":LABEL_RULE},
                        OUT/"labeling_models"/f"{name}.joblib",compress=3)
            DB_MODELS[name]=density
            model=fit_reason_models(name,"logistic",1000.0)
            extra={"labeler_family":"dbscan","eps":eps,"eps_quantile":quantile,"min_samples":min_samples,
                   "clusters":len(unique),"noise_trials":int((ids==-1).sum())}
            row=record_candidate(name+"__logistic_C1000",model,name,"label_generation",{"C":1000.0},extra)
            LABELER_ROWS.append({**row,**extra})

In [8]:
run_dbscan(5)

dbscan_m5_q25__logistic_C1000: sample F1=0.9103, macro F1=0.8114, change=+0.0043


dbscan_m5_q50__logistic_C1000: sample F1=0.8889, macro F1=0.7896, change=-0.0171


dbscan_m5_q75__logistic_C1000: sample F1=0.8932, macro F1=0.8043, change=-0.0128


In [9]:
run_dbscan(10)

dbscan_m10_q25__logistic_C1000: sample F1=0.8974, macro F1=0.8039, change=-0.0085


dbscan_m10_q50__logistic_C1000: sample F1=0.8932, macro F1=0.7987, change=-0.0128


dbscan_m10_q75__logistic_C1000: sample F1=0.8932, macro F1=0.8043, change=-0.0128


## Graph-based LabelSpreading

Use a nearest-neighbor graph with 5,10,20 neighbors and alpha=0.2 or 0.8. Fit one binary graph model per reason on all 1,171 training trials, converting masked labels to sklearn's internal -1 sentinel. Use maximum 1,000 iterations and tolerance 0.0001. Original known targets remain unchanged in generated datasets even though LabelSpreading uses soft clamping internally.

For augmentation, require a graph component with at least 3 known-label trials, reachable nonzero label mass, the same per-reason confidence thresholds, and the same influence budgets. Evaluate C=1000 logistic regression on those generated labels.

Also evaluate the graph model directly, without retraining logistic regression. Direct prediction sums propagated class distributions of training neighbors and normalizes them, matching sklearn's knn-kernel prediction. If a reason has zero reachable mass, use its known-only logistic probability instead of inventing a negative label. Direct graph predictions and augmentation are separate methods; the direct model does not use pseudo-label weight budgets.

In [10]:
def run_graph(neighbors,alpha):
    name=f"graph_n{neighbors}_a{safe_name(alpha)}"
    with threadpool_limits(limits=1):
        nn=NearestNeighbors(n_neighbors=neighbors,algorithm="brute",n_jobs=1).fit(XC)
        adjacency=nn.kneighbors_graph(XC,mode="connectivity")
        component_count,component=connected_components(adjacency.maximum(adjacency.T),directed=False)
        component_support=np.bincount(component[known],minlength=component_count)
        support=component_support[component]
        probabilities=np.zeros((len(train),5));reachable=np.zeros((len(train),5),dtype=bool)
        graphs=[]
        for j in range(5):
            labels=np.full(len(train),-1,dtype=int);labels[known]=original_y[known,j]
            graph=LabelSpreading(kernel="knn",n_neighbors=neighbors,alpha=alpha,max_iter=1000,tol=1e-4,n_jobs=1)
            with warnings.catch_warnings():
                warnings.simplefilter("error",ConvergenceWarning)
                graph.fit(XC,labels)
            mass=graph.label_distributions_.sum(axis=1)
            reachable[:,j]=np.isfinite(graph.label_distributions_).all(axis=1)&(mass>0)
            pos=list(graph.classes_).index(1)
            probabilities[:,j]=np.divide(graph.label_distributions_[:,pos],mass,out=np.zeros(len(train)),where=mass>0)
            graphs.append(graph)
        targets,weights=targets_from_probabilities(probabilities,support,reachable)
        save_dataset(name,targets,weights,np.where(reachable,probabilities,np.nan),component,support)
        pd.DataFrame({"component":np.arange(component_count),"known_support":component_support,
                      "trials":np.bincount(component,minlength=component_count)}).to_csv(OUT/"clusters"/f"{name}.csv",index=False)
        graph_model={"kind":"graph_direct","graphs":graphs,"scaler":cluster_scaler,"fallback_model":baseline,
                     "features":FEATURES,"reasons":REASONS,"dataset":name,"parameters":{"neighbors":neighbors,"alpha":alpha}}
        GRAPH_MODELS[name]=graph_model
        joblib.dump(graph_model,OUT/"labeling_models"/f"{name}.joblib",compress=3)
        extra={"labeler_family":"label_spreading","neighbors":neighbors,"alpha":alpha,
               "graph_components":component_count,"training_reason_slots_without_label_mass":int((~reachable).sum()),
               "maximum_iterations_used":max(int(g.n_iter_) for g in graphs)}
        model=fit_reason_models(name,"logistic",1000.0)
        row=record_candidate(name+"__logistic_C1000",model,name,"label_generation",{"C":1000.0},extra)
        LABELER_ROWS.append({**row,**extra})
        _,fallback_slots=predict_graph_model(graph_model,X_valid)
        record_candidate(name+"__direct",graph_model,name,"graph_direct",graph_model["parameters"],
                         {**extra,"validation_reason_slots_using_supervised_fallback":fallback_slots})

In [11]:
run_graph(5,0.2)

graph_n5_a0p2__logistic_C1000: sample F1=0.8504, macro F1=0.6578, change=-0.0556


graph_n5_a0p2__direct: sample F1=0.8077, macro F1=0.5360, change=-0.0983


In [12]:
run_graph(5,0.8)

graph_n5_a0p8__logistic_C1000: sample F1=0.8632, macro F1=0.6710, change=-0.0427


graph_n5_a0p8__direct: sample F1=0.7949, macro F1=0.5188, change=-0.1111


In [13]:
run_graph(10,0.2)

graph_n10_a0p2__logistic_C1000: sample F1=0.8291, macro F1=0.5856, change=-0.0769


graph_n10_a0p2__direct: sample F1=0.8205, macro F1=0.5402, change=-0.0855


In [14]:
run_graph(10,0.8)

graph_n10_a0p8__logistic_C1000: sample F1=0.8269, macro F1=0.6673, change=-0.0791


graph_n10_a0p8__direct: sample F1=0.7564, macro F1=0.4551, change=-0.1496


In [15]:
run_graph(20,0.2)

graph_n20_a0p2__logistic_C1000: sample F1=0.8632, macro F1=0.7199, change=-0.0427


graph_n20_a0p2__direct: sample F1=0.7821, macro F1=0.4767, change=-0.1239


In [16]:
run_graph(20,0.8)

graph_n20_a0p8__logistic_C1000: sample F1=0.8504, macro F1=0.7384, change=-0.0556


graph_n20_a0p8__direct: sample F1=0.7308, macro F1=0.3821, change=-0.1752


## Combine complementary label generators

After inspecting the first results, test one additional per-reason hybrid. The frozen GMM contributes generated Recruitment labels, while the leading DBSCAN contributes generated Efficacy labels. For each reason, use DBSCAN if it supplies generated labels and GMM does not; otherwise retain GMM if it supplies labels, and use known-only targets if neither does. If both supply a reason, prefer the incumbent GMM rather than searching another mixing parameter.

This follow-up was motivated by complementary label availability, not a separate search over per-reason validation winners. Keep each source's target labels and weight budgets exactly, then fit the same five logistic regressions at C=1000. Because all classifiers share the same feature scaler and per-reason fitting procedure, their probabilities should match the corresponding source classifier columns.

In [17]:
dbscan_candidates=pd.DataFrame(LABELER_ROWS)
dbscan_best=dbscan_candidates.loc[dbscan_candidates.labeler_family=="dbscan"].sort_values(
    ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
dbscan_dataset=DATASETS[dbscan_best.dataset]
gmm_dataset=DATASETS["frozen_gmm_tied_k10"]
hybrid_y=known_y.copy();hybrid_w=known_w.copy();hybrid_sources={}
for j,r in enumerate(REASONS):
    gm=bool((gmm_dataset["weights"][~known,j]>0).any())
    db=bool((dbscan_dataset["weights"][~known,j]>0).any())
    if gm:chosen=gmm_dataset;hybrid_sources[r]="frozen_gmm_tied_k10"
    elif db:chosen=dbscan_dataset;hybrid_sources[r]=dbscan_best.dataset
    else:chosen=DATASETS["known_only"];hybrid_sources[r]="known_only"
    hybrid_y[:,j]=chosen["targets"][:,j]
    hybrid_w[:,j]=chosen["weights"][:,j]
save_dataset("hybrid_gmm_dbscan",hybrid_y,hybrid_w)
with threadpool_limits(limits=1):
    hybrid_model=fit_reason_models("hybrid_gmm_dbscan","logistic",1000.0)
    hybrid_probabilities=predict_reason_models(hybrid_model,X_valid)
    db_probabilities=predict_candidate(MODELS[dbscan_best.experiment],X_valid)
    gm_probabilities=predict_reason_models(incumbent,X_valid)
    known_probabilities=baseline.predict_proba(X_valid)
for j,r in enumerate(REASONS):
    expected=(gm_probabilities if hybrid_sources[r]=="frozen_gmm_tied_k10" else
              db_probabilities if hybrid_sources[r]==dbscan_best.dataset else known_probabilities)
    np.testing.assert_allclose(hybrid_probabilities[:,j],expected[:,j],rtol=1e-8,atol=1e-10)
record_candidate("hybrid_gmm_dbscan__logistic_C1000",hybrid_model,"hybrid_gmm_dbscan","hybrid_label_generation",
                 {"C":1000.0,"label_sources":hybrid_sources})
print("Hybrid label sources:",hybrid_sources)

hybrid_gmm_dbscan__logistic_C1000: sample F1=0.8889, macro F1=0.7937, change=-0.0171


Hybrid label sources: {'Safety': 'known_only', 'Funding': 'known_only', 'Efficacy': 'dbscan_m5_q25', 'Recruitment': 'frozen_gmm_tied_k10', 'Other': 'known_only'}


## Small crossover check

Select one label-generated dataset from DBSCAN and one from LabelSpreading by their C=1000 logistic validation results. Re-evaluate each with the best RBF-SVM parameter setting and best GaussianNB setting from the initial classifier comparison. This adds four controlled comparisons; it does not search every classifier/labeler combination.

In [18]:
initial=pd.DataFrame(RESULTS)
best_parameters={}
for kind in ["rbf_svm","gaussian_nb"]:
    row=initial.loc[(initial.classifier==kind)&(initial.category=="supervised_comparison")].sort_values(
        ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
    best_parameters[kind]=json.loads(row.parameters)
labeler_comparison=pd.DataFrame(LABELER_ROWS)
labeler_shortlist=[]
for family in ["dbscan","label_spreading"]:
    row=labeler_comparison.loc[labeler_comparison.labeler_family==family].sort_values(
        ["sample_f1","experiment"],ascending=[False,True],kind="stable").iloc[0]
    labeler_shortlist.append(row.dataset)
with threadpool_limits(limits=1):
    for dataset_name in labeler_shortlist:
        for kind,parameters in best_parameters.items():
            name=f"{dataset_name}__crossover_{kind}"
            model=fit_additional_classifier(dataset_name,kind,parameters)
            record_candidate(name,model,dataset_name,"classifier_crossover",parameters)
# Follow-up requested after the initial method comparison: KNN on the new label sets.
knn_followup_datasets=list(dict.fromkeys(labeler_shortlist+["hybrid_gmm_dbscan"]))
knn_neighbor_counts=[1,3,7,15]
knn_dbscan_reference=float(pd.DataFrame(LABELER_ROWS).loc[lambda d:d.labeler_family=="dbscan","sample_f1"].max())
with threadpool_limits(limits=1):
    for dataset_name in knn_followup_datasets:
        for neighbors in knn_neighbor_counts:
            name=f"{dataset_name}__weighted_knn_k{neighbors}"
            model=fit_reason_models(dataset_name,"weighted_knn",neighbors)
            row=record_candidate(name,model,dataset_name,"knn_followup",{"neighbors":neighbors})
            row["delta_vs_dbscan_lr"]=row["sample_f1"]-knn_dbscan_reference
results=pd.DataFrame(RESULTS)
results.to_csv(OUT/"new_comparison.csv",index=False)
results.loc[results.category=="knn_followup"].to_csv(OUT/"knn_comparison.csv",index=False)
labeler_comparison.to_csv(OUT/"labeler_comparison.csv",index=False)
pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
all_results=pd.concat([pd.DataFrame(reference_rows),results],ignore_index=True)
all_results.to_csv(OUT/"comparison.csv",index=False)
ranked=results.sort_values(["sample_f1","experiment"],ascending=[False,True],kind="stable")
best_new=ranked.iloc[0]
best_model_name=best_new.experiment if best_new.sample_f1>incumbent_metrics["sample_f1"]+1e-12 else "current_validation_best"
print("Leading model after this round:",best_model_name)
display(ranked.head(12)[["experiment","classifier","sample_f1","macro_f1","delta_vs_current_best"]].round(4))

dbscan_m5_q25__crossover_rbf_svm: sample F1=0.8376, macro F1=0.6656, change=-0.0684


dbscan_m5_q25__crossover_gaussian_nb: sample F1=0.7201, macro F1=0.6713, change=-0.1859


graph_n20_a0p2__crossover_rbf_svm: sample F1=0.8291, macro F1=0.5876, change=-0.0769


graph_n20_a0p2__crossover_gaussian_nb: sample F1=0.8295, macro F1=0.7200, change=-0.0765


dbscan_m5_q25__weighted_knn_k1: sample F1=0.8547, macro F1=0.6690, change=-0.0513


dbscan_m5_q25__weighted_knn_k3: sample F1=0.8504, macro F1=0.5864, change=-0.0556


... (8 outputs left out)

hybrid_gmm_dbscan__weighted_knn_k7: sample F1=0.8376, macro F1=0.5478, change=-0.0684


hybrid_gmm_dbscan__weighted_knn_k15: sample F1=0.8462, macro F1=0.5527, change=-0.0598


Leading model after this round: dbscan_m5_q25__logistic_C1000


,experiment,classifier,sample_f1,macro_f1,delta_vs_current_best
24,dbscan_m5_q25__logistic_C1000,logistic,0.9103,0.8114,0.0043
27,dbscan_m10_q25__logistic_C1000,logistic,0.8974,0.8039,-0.0085
28,dbscan_m10_q50__logistic_C1000,logistic,0.8932,0.7987,-0.0128
29,dbscan_m10_q75__logistic_C1000,logistic,0.8932,0.8043,-0.0128
26,dbscan_m5_q75__logistic_C1000,logistic,0.8932,0.8043,-0.0128
25,dbscan_m5_q50__logistic_C1000,logistic,0.8889,0.7896,-0.0171
42,hybrid_gmm_dbscan__logistic_C1000,logistic,0.8889,0.7937,-0.0171
16,frozen_gmm_tied_k10__rbf_svm_C10p0_gammascale,rbf_svm,0.8675,0.7206,-0.0385
38,graph_n20_a0p2__logistic_C1000,logistic,0.8632,0.7199,-0.0427
32,graph_n5_a0p8__logistic_C1000,logistic,0.8632,0.6710,-0.0427


In [19]:
# Verify the graph predictor against sklearn on rows where normalization is defined.
first_graph=next(iter(GRAPH_MODELS.values()))
scaled_valid=first_graph["scaler"].transform(X_valid)
for j,graph in enumerate(first_graph["graphs"]):
    ids=graph.nn_fit.kneighbors(scaled_valid,return_distance=False)
    mass=graph.label_distributions_[ids].sum(axis=1)
    good=mass.sum(axis=1)>0
    if good.any():
        native=graph.predict_proba(scaled_valid[good])[:,list(graph.classes_).index(1)]
        custom=predict_graph_model(first_graph,X_valid[good])[0][:,j]
        np.testing.assert_allclose(native,custom,rtol=1e-12,atol=1e-12)

selected_names=[best_new.experiment]
best_label_aug=ranked.loc[ranked.category=="label_generation"].iloc[0]
if best_label_aug.experiment not in selected_names:selected_names.append(best_label_aug.experiment)
for name in selected_names:
    reloaded=joblib.load(OUT/"classifier_models"/f"{name}.joblib")
    expected=predict_candidate(MODELS[name],X_valid)
    np.testing.assert_allclose(predict_candidate(reloaded,X_valid),expected,rtol=1e-12,atol=1e-12)
    if reloaded["kind"]!="graph_direct":
        frame=pd.read_csv(OUT/"datasets"/f"{reloaded['dataset']}.csv",float_precision="round_trip")
        assert frame.ID.tolist()==train.ID.tolist()
        for j,r in enumerate(REASONS):
            np.testing.assert_array_equal(frame[r].to_numpy(),DATASETS[reloaded["dataset"]]["targets"][:,j])
            np.testing.assert_allclose(frame[f"weight_{r}"].to_numpy(),DATASETS[reloaded["dataset"]]["weights"][:,j],atol=1e-14)
        # Independently refit from exported features, labels, and weights.
        from sklearn.base import clone
        replay={**reloaded,"estimators":[]}
        saved_scaled=reloaded["scaler"].transform(frame[FEATURES].to_numpy())
        with threadpool_limits(limits=1):
            for j,r in enumerate(REASONS):
                y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy()
                included=w>0
                if reloaded["kind"]=="weighted_knn":
                    estimator={"neighbors":NearestNeighbors(n_neighbors=min(int(reloaded["parameter"]),int(included.sum())),algorithm="brute").fit(saved_scaled[included]),
                               "target":y[included],"sample_weight":w[included]}
                else:
                    estimator=clone(reloaded["estimators"][j])
                    with warnings.catch_warnings():
                        warnings.simplefilter("error",ConvergenceWarning)
                        estimator.fit(saved_scaled[included],y[included],sample_weight=w[included]*known.sum()/w[included].sum())
                replay["estimators"].append(estimator)
        np.testing.assert_array_equal(label_predictions(predict_candidate(replay,X_valid))[0],label_predictions(expected)[0])
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

record={"features":FEATURES,"deep_learning":False,"new_features_added":False,
    "svm_grid":{"C":SVM_C,"gamma":SVM_GAMMA,"calibration":"sigmoid; 3 training-only stratified folds; ensemble=False"},
    "gaussian_nb_smoothing":NB_SMOOTHING,"dbscan_min_samples":DBSCAN_MIN_SAMPLES,
    "dbscan_radius_quantiles":DBSCAN_RADIUS_QUANTILES,"graph_neighbors":GRAPH_NEIGHBORS,"graph_alpha":GRAPH_ALPHA,
    "graph_max_iter":1000,"graph_tol":1e-4,"graph_direct_fallback":"known-only logistic for zero label mass",
    "knn_followup_datasets":knn_followup_datasets,"knn_followup_neighbors":knn_neighbor_counts,
    "knn_followup_reference_sample_f1":knn_dbscan_reference,
    "label_rule":LABEL_RULE,"classifier_scaler_fit_rows":157,"graph_cluster_scaler_fit_rows":1171,
    "fitting_weight_normalization":157,"threshold":0.5,"nonempty_fallback":True,
    "validation_rows":78,"validation_used_for_fit":False,"best_new":best_new.experiment,
    "best_overall":best_model_name,"hybrid_label_sources":hybrid_sources,"labeler_crossover_datasets":labeler_shortlist,"crossover_parameters":best_parameters,
    "protected_input_sha256":protected_hashes,
    "dataset_sha256":{str(q.relative_to(OUT)):hashlib.sha256(q.read_bytes()).hexdigest() for q in sorted((OUT/"datasets").glob("*.csv"))},
    "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},
    "kaggle_scores_new_methods":None,"reported_kaggle_best":0.87393,
}
(OUT/"settings.json").write_text(json.dumps(record,indent=2)+"\n")
lines=["# Additional classical ML methods","",
    "All models use only the same 150 embeddings, the same 78 validation trials, threshold 0.5, and nonempty fallback. "
    "No neural networks, extra PCA, year, text, or drug features were used. No Kaggle submission was made.","",
    f"Current validation reference: **{incumbent_metrics['sample_f1']:.4f}**. Best new model: **{best_new.experiment}**, "
    f"sample-wise F1 **{best_new.sample_f1:.4f}**. Leading model after this round: **{best_model_name}**.","",
    "## Best result by method","",
    "| Method | Configuration | Sample-wise F1 | Macro F1 | Change from 0.9060 reference |","|---|---|---:|---:|---:|"]
for classifier in ["rbf_svm","gaussian_nb","logistic","graph_direct","weighted_knn"]:
    group=ranked.loc[ranked.classifier==classifier]
    if len(group):
        row=group.iloc[0]
        lines.append(f"| {classifier} | {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.delta_vs_current_best:+.4f} |")
lines.extend(["","## Leading new configurations","",
    "| Configuration | Sample-wise F1 | Macro F1 |","|---|---:|---:|"])
for row in ranked.head(12).itertuples(index=False):lines.append(f"| {row.experiment} | {row.sample_f1:.4f} | {row.macro_f1:.4f} |")
lines.extend(["","## Method details and limits","",
    "RBF SVM compares C=0.1,1,10 and gamma=0.001,scale,0.03 on known-only and frozen GMM labels. "
    "Its probabilities are calibrated with sigmoid calibration and three stratified training-only folds; validation is excluded. "
    "GaussianNB compares smoothing 1e-9,0.001,0.1 on the same two datasets. It accepts continuous signed features; "
    "MultinomialNB is not appropriate for these embeddings.","",
    "DBSCAN compares min_samples=5,10 and radii taken from the 25th/50th/75th percentiles of training neighbor distances. "
    "Noise is excluded from generated targets. LabelSpreading compares 5,10,20 neighbors and alpha=0.2,0.8, "
    "one graph per reason. Missing labels are -1 internally for sklearn and remain -100 in saved uncertain targets. "
    "For augmentation, graph components need at least 3 known-label members and nonzero propagated mass. "
    "Both labelers reuse the fixed confidence thresholds, original-label preservation, and generated-influence budgets. "
    "Their augmented datasets train C=1000 logistic regression.","",
    "Direct graph prediction is also evaluated from each LabelSpreading fit. It sums class distributions over training neighbors "
    "and uses known-only logistic probabilities only if propagated mass is zero. It is separate from generating labels and fitting LR. "
    "The direct graph model uses the training-only graph scaler and does not use augmentation weight budgets.","",
    "A small crossover refits the best SVM and GaussianNB parameter settings on the best DBSCAN and LabelSpreading datasets. "
    "One adaptive follow-up combines GMM-generated Recruitment labels with DBSCAN-generated Efficacy labels, retaining known-only labels for the other reasons and preserving all source weights. "
    "This staged search does not establish a global maximum. All new scores are validation selections, not independently measured Kaggle scores. "
    "The reported best submitted Kaggle model remains K=2 without year or Safety boost at 0.87393.","",
    "## Artifacts and controls","",
    "`datasets/` and `label_audits/` preserve each training table and target/weight mask; `clusters/` records density or graph components. "
    "`labeling_models/` and `classifier_models/` save fitted models. `validation_predictions/` preserves probabilities and labels. "
    "`comparison.csv`, `new_comparison.csv`, `labeler_comparison.csv`, and `label_counts.csv` retain all results. "
    "`settings.json` records grids, selected models, source/data checksums, and software versions.","",
    "Controls reproduced the old baseline, preserved all known labels and protected artifacts, excluded noise and unreachable graph nodes "
    "from generated targets, matched direct graph predictions to sklearn where defined, reloaded selected models with matching predictions, and refitted selected classifiers from their saved datasets.",""])
hybrid_result=results.set_index("experiment").loc["hybrid_gmm_dbscan__logistic_C1000"]
lines.extend([
    "", "## Complementary-label hybrid", "",
    f"The one additional GMM-Recruitment/DBSCAN-Efficacy hybrid scored sample-wise F1 **{hybrid_result.sample_f1:.4f}** "
    f"and macro F1 **{hybrid_result.macro_f1:.4f}**, below both leading individual label-generation approaches. "
    "The source-probability controls confirmed that each reason's probabilities matched its selected source classifier. "
    "Sample-wise F1 and the nonempty fallback couple the five outputs, so changes that help separately need not help when combined. "
    "Retain this negative result; do not use the hybrid as the leading candidate.",
])
lines.extend(["","## KNN follow-up on the new label sets","",
    "The earlier KNN results used older datasets. A separate follow-up tests k=1,3,7,15 on the best DBSCAN and LabelSpreading datasets "
    "and the complementary-label hybrid, using the same frozen classifier scaler and distance/sample weights.","",
    "| Dataset | Best neighbor count | Sample-wise F1 | Macro F1 | Difference from DBSCAN + LR |",
    "|---|---:|---:|---:|---:|"])
knn_rows=results.loc[results.category=="knn_followup"]
for dataset_name in knn_followup_datasets:
    row=knn_rows.loc[knn_rows.dataset==dataset_name].sort_values(["sample_f1","experiment"],ascending=[False,True]).iloc[0]
    neighbors=json.loads(row.parameters)["neighbors"]
    lines.append(f"| {dataset_name} | {neighbors} | {row.sample_f1:.4f} | {row.macro_f1:.4f} | {row.delta_vs_dbscan_lr:+.4f} |")
lines.extend(["","All twelve follow-up configurations are saved in `knn_comparison.csv`. "
    "The results only compare these tested settings; earlier losses alone did not establish that KNN could not benefit from new labels."])
summary="\n".join(lines).rstrip()+"\n"
(OUT/"SUMMARY.md").write_text(summary)
print(summary)

# Additional classical ML methods

All models use only the same 150 embeddings, the same 78 validation trials, threshold 0.5, and nonempty fallback. No neural networks, extra PCA, year, text, or drug features were used. No Kaggle submission was made.

Current validation reference: **0.9060**. Best new model: **dbscan_m5_q25__logistic_C1000**, sample-wise F1 **0.9103**. Leading model after this round: **dbscan_m5_q25__logistic_C1000**.

## Best result by method

| Method | Configuration | Sample-wise F1 | Macro F1 | Change from 0.9060 reference |
|---|---|---:|---:|---:|
| rbf_svm | frozen_gmm_tied_k10__rbf_svm_C10p0_gammascale | 0.8675 | 0.7206 | -0.0385 |
| gaussian_nb | graph_n20_a0p2__crossover_gaussian_nb | 0.8295 | 0.7200 | -0.0765 |
| logistic | dbscan_m5_q25__logistic_C1000 | 0.9103 | 0.8114 | +0.0043 |
| graph_direct | graph_n10_a0p2__direct | 0.8205 | 0.5402 | -0.0855 |
| weighted_knn | dbscan_m5_q25__weighted_knn_k1 | 0.8547 | 0.6690 | -0.0513 |

## Leading new configurations

## Prepare a candidate without using a Kaggle slot

If a new model beats the 0.9060 validation reference, export its test predictions for later manual review. Preparing this file does not submit it. The previously reported Kaggle best remains 0.87393 and its Git tag stays unchanged. Keep this candidate available while deciding whether to test additional features before the final daily submission.

In [20]:
new_candidate_record=None
if best_model_name!="current_validation_best":
    candidate=joblib.load(OUT/"classifier_models"/f"{best_model_name}.joblib")
    test=pd.read_csv(INPUT_DIR/"part1_test.csv")
    sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    probability=predict_candidate(candidate,test[FEATURES].to_numpy())
    predicted,fallback=label_predictions(probability)
    frame=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in predicted]})
    frame=frame.set_index("ID").loc[sample.ID].reset_index()
    assert frame.shape==(312,2) and frame.Predicted.str.len().gt(0).all()
    assert frame.Predicted.map(lambda s:set(s.split()).issubset(REASONS)).all()
    path=OUT/"submissions"/f"part1_{best_model_name.replace('__','_')}_submission.csv"
    frame.to_csv(path,index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path),frame)
    selected=results.set_index("experiment").loc[best_model_name]
    new_candidate_record={"experiment":best_model_name,"validation_sample_f1":float(selected.sample_f1),
        "validation_macro_f1":float(selected.macro_f1),"test_rows":len(frame),"test_fallback_trials":int(fallback.sum()),
        "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
        "model_file":f"classifier_models/{best_model_name}.joblib",
        "model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best_model_name}.joblib").read_bytes()).hexdigest(),
        "kaggle_score":None,"kaggle_status":"prepared locally; not submitted","reported_kaggle_comparator":0.87393}
    (OUT/"submission_manifest.json").write_text(json.dumps(new_candidate_record,indent=2)+"\n")
    summary_path=OUT/"SUMMARY.md"
    text=summary_path.read_text().split("## Prepared candidate")[0].rstrip()
    text+=(f"\n\n## Prepared candidate\n\n`{path.relative_to(OUT)}` contains 312 predictions from **{best_model_name}**. "
           f"Validation sample-wise F1: **{selected.sample_f1:.4f}**; macro F1: **{selected.macro_f1:.4f}**. "
           "The file is prepared locally, not submitted. No Kaggle slot was used, and the submitted-best tag is unchanged. "
           "The model and file checksums are recorded in `submission_manifest.json`.\n")
    summary_path.write_text(text)
    print(json.dumps(new_candidate_record,indent=2))
else:
    print("No new candidate exceeded the current validation reference; retain its existing submission file.")
for relative,digest in protected_hashes.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest

{
  "experiment": "dbscan_m5_q25__logistic_C1000",
  "validation_sample_f1": 0.9102564102564102,
  "validation_macro_f1": 0.811421460284173,
  "test_rows": 312,
  "test_fallback_trials": 18,
  "submission_file": "submissions/part1_dbscan_m5_q25_logistic_C1000_submission.csv",
  "submission_sha256": "b4f8e154c6105698d6c7fac2a34d3c0e1ecd3dcd095bd2d8634ee6608d336dba",
  "model_file": "classifier_models/dbscan_m5_q25__logistic_C1000.joblib",
  "model_sha256": "522ae60a545a600b9f993ce4fac66d55e37c8ae8c4d7ba1dbe7dfa72f82598a3",
  "kaggle_score": null,
  "kaggle_status": "prepared locally; not submitted",
  "reported_kaggle_comparator": 0.87393
}
